# StormLead — Run A State & Output Schemas

Flow: user types a city -> weather MCP tool (`get_current_weather`) checks
conditions for that city -> if `StormEvent.qualifies` (storm condition code
or wind over `WIND_THRESHOLD_MPH`), the city name is passed to the DB tool
(`get_customers_by_city`) to pull leads for the personalizer.

- `RunAState`: the LangGraph state (TypedDict) passed node-to-node.
- `StormEvent`, `Lead`, `StormContext`, `EmailDraft`: Pydantic models for structured node outputs.

In [18]:
from typing import TypedDict, Annotated, Literal, Optional
import operator, uuid
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.postgres import PostgresSaver   # or AsyncPostgresSaver if async

import psycopg
from psycopg.rows import dict_row
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

import mcp.client.stdio as mcp_stdio

import sys
from pathlib import Path
load_dotenv()
from IPython.display import Image,display

In [19]:
mcp_stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

In [20]:
from typing import Literal

from pydantic import BaseModel, model_validator

WIND_THRESHOLD_MPH = 50.0

# OpenWeather condition-code groups that count as a storm.
# 2xx = thunderstorm, 7xx range includes squall (771) and tornado (781).
THUNDERSTORM_CODES = range(200, 300)
SQUALL_TORNADO_CODES = {771, 781}


def _to_mph(speed: float, units: Literal["m/s", "mph"]) -> float:
    """OpenWeather returns m/s on metric, mph on imperial. Normalize to mph."""
    return speed * 2.23694 if units == "m/s" else speed


class StormEvent(BaseModel):
    """One detected weather event for a city. The monitor node produces this
    from the openweather MCP tool's `get_current_weather` output.

    Built around OpenWeather data: no hail available, so the trigger is
    storm condition codes OR high wind. `qualifies` is computed, never hand-set.
    """

    city: str
    condition_code: int          # OpenWeather weather.id
    condition_text: str          # human-readable, e.g. "Thunderstorm"
    wind_speed: float            # raw value from OpenWeather
    wind_units: Literal["m/s", "mph"]   # store units so the threshold is unambiguous
    date: str
    qualifies: bool = False

    @property
    def wind_mph(self) -> float:
        return _to_mph(self.wind_speed, self.wind_units)

    @model_validator(mode="after")
    def _set_qualifies(self) -> "StormEvent":
        is_storm = (
            self.condition_code in THUNDERSTORM_CODES
            or self.condition_code in SQUALL_TORNADO_CODES
        )
        self.qualifies = is_storm or self.wind_mph >= WIND_THRESHOLD_MPH
        return self


class Lead(BaseModel):
    """One targeted customer + property record for the personalizer.

    Field names/types mirror the `customers` table (db/schema.sql) directly,
    so a DB row can be validated straight into this model with no remapping.
    `customer_id` is carried through so drafts/tracking can reference it.
    """

    customer_id: int
    first_name: str
    last_name: str
    email: str
    city: str
    state: str
    zip: str
    roof_material: str
    roof_age_years: int
    insurance_carrier: str | None = None


class StormContext(BaseModel):
    """Compact storm summary carried with a lead into later runs.

    Also stored in the tracking ledger so Run B's qualifier can read it.
    """

    city: str
    condition_text: str
    wind_mph: float
    date: str


class EmailDraft(BaseModel):
    """One personalized draft. The personalizer produces a list; the human gate
    approves; the sender consumes."""

    customer_id: int
    to_email: str
    subject: str
    body: str
    storm_context: StormContext
    gmail_thread_id: str | None = None

In [21]:
import operator
from typing import Annotated, TypedDict


class RunAState(TypedDict):
    # trigger/input -- city name(s) typed by the user
    service_cities: list[str]

    # monitor -- StormEvents built from OpenWeather (condition code + wind), one per city
    storm_events: list[StormEvent]
    qualifying_cities: list[str]

    # targeting -- Leads pulled from the DB (already opted-in, cooldown-filtered)
    leads: list[Lead]

    # personalize
    drafts: list[EmailDraft]

    # human gate
    approved: bool
    approved_drafts: list[EmailDraft]

    # sender -- {customer_id, gmail_thread_id, status}
    sent: list[dict]

    # accumulates across any node
    errors: Annotated[list[str], operator.add]

## Weather MCP (custom, OpenWeather-backed)

Custom stdio MCP server at `mcp_servers/weather_server.py` — wraps `GET https://api.openweathermap.org/data/2.5/weather` and exposes a single tool, `get_current_weather(location, units)`, where `location` is a city name (e.g. `"Lahore"`, `"Paris,FR"`) or a zip/postal code (e.g. `"10001"`). Returns conditions, now/high/low temp, pressure, humidity, feels-like, wind speed/deg, and sunrise/sunset (unix time).

In [22]:
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
if WEATHER_API_KEY:
    print("True")

True


In [23]:
WEATHER_MCP_SERVER = (Path.cwd().parent / "mcp_servers" / "weather_server.py").resolve()
assert WEATHER_MCP_SERVER.exists(), f"missing {WEATHER_MCP_SERVER}"

sys.path.append(str(Path.cwd().parent))  # so `backend` is importable from notebook/
from backend.gmail_client import get_valid_access_token

# gmailmcp.googleapis.com/mcp/v1 is a real Google-hosted Gmail MCP server (verified
# directly with an MCP `initialize` + `tools/list` call). It exposes create_draft,
# list_drafts, search_threads, get_thread, get_message, and label tools -- but NO
# send tool: a human has to open the draft in Gmail and click Send themselves.
# Auth is a plain Bearer header (langchain-mcp-adapters' StreamableHttpConnection
# has no `oauth` field at all -- that's what raised the earlier TypeError). The
# token below is fetched fresh from the gmail_credentials table (refreshed via the
# stored refresh_token if the access token had expired) right before building the
# client -- if a session runs long past the token's ~1hr lifetime, re-run this cell.
GMAIL_ACCESS_TOKEN = get_valid_access_token()

client = MultiServerMCPClient(
    {
        "openweather": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(WEATHER_MCP_SERVER)],
            "env": {
                "OPENWEATHER_API_KEY": WEATHER_API_KEY
            },
        },
        "gmail": {
            "transport": "streamable_http",
            "url": "https://gmailmcp.googleapis.com/mcp/v1",
            "headers": {
                "Authorization": f"Bearer {GMAIL_ACCESS_TOKEN}"
            },
        },
    }
)

ImportError: cannot import name 'get_valid_access_token' from 'backend.gmail_client' (d:\Agentic&MCP\Strom Lead Engine\backend\gmail_client.py)

In [17]:
tools = await client.get_tools()
print(tools)

TypeError: _create_streamable_http_session() got an unexpected keyword argument 'oauth'

#tool to get data of customers from a city

In [8]:
from langchain_core.tools import tool


@tool
async def get_customers_by_city(city: str) -> list[Lead]:
    """Look up opted-in customers in a given city.

    Returns one Lead per matching customer: customer_id, first_name,
    last_name, email, city, state, zip, roof_material, roof_age_years,
    and insurance_carrier.
    """
    query = """
        SELECT customer_id, first_name, last_name, email, city, state, zip,
               roof_material, roof_age_years, insurance_carrier
        FROM customers
        WHERE city ILIKE %s AND marketing_opt_in = true
    """
    async with await psycopg.AsyncConnection.connect(
        os.environ["DATABASE_URL"], row_factory=dict_row
    ) as conn:
        async with conn.cursor() as cur:
            await cur.execute(query, (city,))
            rows = await cur.fetchall()
            return [Lead(**row) for row in rows]

# weather and customer agent

In [9]:
import json
from datetime import datetime, timezone


async def weather_monitor_agent(state: RunAState) -> dict:
    """Monitor node: checks each service city for storm conditions via the
    openweather MCP tool and builds a StormEvent per city.

    A city ends up in `qualifying_cities` when `StormEvent.qualifies` is True
    (storm condition code or wind >= WIND_THRESHOLD_MPH). Per-city failures
    are appended to `errors` instead of aborting the whole run.
    """
    storm_events: list[StormEvent] = []
    qualifying_cities: list[str] = []
    errors: list[str] = []

    for city in state["service_cities"]:
        try:
            raw = await weather_tool.ainvoke({"location": city, "units": "imperial"})
            payload = json.loads(raw[0]["text"])

            event = StormEvent(
                city=city,
                condition_code=payload["condition_code"],
                condition_text=payload["conditions"],
                wind_speed=payload["wind_speed"],
                wind_units="mph" if payload["units"] == "imperial" else "m/s",
                date=datetime.fromtimestamp(payload["observed_at"], tz=timezone.utc).date().isoformat(),
            )
        except Exception as exc:
            errors.append(f"{city}: weather lookup failed - {exc}")
            continue

        storm_events.append(event)
        if event.qualifies:
            qualifying_cities.append(city)

    return {
        "storm_events": storm_events,
        "qualifying_cities": qualifying_cities,
        "errors": errors,
    }

# targeting agent

In [10]:
async def targeting_agent(state: RunAState) -> dict:
    """Targeting node: for each qualifying city (from the monitor node), pulls
    opted-in customers via the `get_customers_by_city` DB tool and gathers
    them into `leads` for the personalizer.
    """
    leads: list[Lead] = []
    errors: list[str] = []

    for city in state["qualifying_cities"]:
        try:
            city_leads = await get_customers_by_city.ainvoke({"city": city})
        except Exception as exc:
            errors.append(f"{city}: customer lookup failed - {exc}")
            continue

        leads.extend(city_leads)

    return {
        "leads": leads,
        "errors": errors,
    }

# personalizer agent (email drafts)

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage


class _EmailContent(BaseModel):
    """Slim structured-output schema for the LLM call — the node wraps this
    into a full EmailDraft (which also needs customer_id/storm_context)."""

    subject: str = Field(description="Short, professional subject line referencing the storm and check-in purpose")
    body: str = Field(description="Full plain-text email body, professional tone, no markdown/bullets")


PERSONALIZER_SYSTEM_PROMPT = """You are a professional, empathetic customer-outreach specialist for a \
roofing company that monitors severe weather across its service area and follows up with homeowners after \
storms. You will be given one homeowner's details and the storm that recently affected their city. Write \
ONE personalized outreach email for that homeowner, following these rules exactly:

1. Tone: warm, professional, concise — a caring check-in, never salesy or alarmist.
2. Open by acknowledging the specific storm/weather condition and date in their city, and briefly express \
sympathy (e.g. "sorry to hear about the recent storm activity in your area").
3. State the purpose plainly: you are reaching out to check whether the storm caused any damage to their \
roof. Do not assume or claim that damage occurred — you are simply checking in.
4. Where it fits naturally, reference their roof material and/or approximate age to make the message feel \
personalized (e.g. older roofs or certain materials can be more vulnerable to storm damage). Do not force \
this if it reads awkwardly for the given details.
5. Call to action: ask them to simply reply to this email if they've noticed any damage (missing/loose \
shingles, leaks, dents, granule loss, etc.) so the company can follow up with more details and arrange a \
free inspection. Make clear that replying is quick and low-effort.
6. If they haven't noticed anything, reassure them no action is needed — but they're welcome to reply \
anyway if they'd like a precautionary inspection.
7. Sign off professionally from "The StormLead Roofing Team".
8. Do NOT invent facts that weren't provided (no insurance claims, no guarantees, no pricing, no specific \
dollar amounts).
9. Length: 120-180 words, plain prose paragraphs only — no markdown, no bullet lists, no placeholder \
brackets like [Name].
"""

PERSONALIZER_USER_TEMPLATE = """Customer: {first_name} {last_name}
Location: {city}, {state}
Roof: {roof_material}, approximately {roof_age_years} years old
Insurance carrier on file: {insurance_carrier}

Storm details for {city}:
- Condition: {condition_text}
- Wind speed: {wind_mph} mph
- Date: {date}

Write the personalized outreach email now, following all system instructions."""


personalizer_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.4).with_structured_output(_EmailContent)


async def personalizer_agent(state: RunAState) -> dict:
    """Personalize node: drafts one storm-damage check-in email per lead,
    using GPT-5.4 mini and that lead's city storm context (from the monitor
    node's `storm_events`).
    """
    events_by_city = {event.city: event for event in state["storm_events"]}

    drafts: list[EmailDraft] = []
    errors: list[str] = []

    for lead in state["leads"]:
        event = events_by_city.get(lead.city)
        if event is None:
            errors.append(f"{lead.customer_id}: no storm event found for city {lead.city}")
            continue

        storm_context = StormContext(
            city=event.city,
            condition_text=event.condition_text,
            wind_mph=round(event.wind_mph, 1),
            date=event.date,
        )

        user_prompt = PERSONALIZER_USER_TEMPLATE.format(
            first_name=lead.first_name,
            last_name=lead.last_name,
            city=lead.city,
            state=lead.state,
            roof_material=lead.roof_material,
            roof_age_years=lead.roof_age_years,
            insurance_carrier=lead.insurance_carrier or "not on file",
            condition_text=storm_context.condition_text,
            wind_mph=storm_context.wind_mph,
            date=storm_context.date,
        )

        try:
            email_content = await personalizer_llm.ainvoke([
                SystemMessage(content=PERSONALIZER_SYSTEM_PROMPT),
                HumanMessage(content=user_prompt),
            ])
        except Exception as exc:
            errors.append(f"{lead.customer_id}: draft generation failed - {exc}")
            continue

        drafts.append(EmailDraft(
            customer_id=lead.customer_id,
            to_email=lead.email,
            subject=email_content.subject,
            body=email_content.body,
            storm_context=storm_context,
        ))

    return {
        "drafts": drafts,
        "errors": errors,
    }

# sender agent (Gmail drafts via MCP -- human sends)

Creates one Gmail draft per approved email via the `gmail` MCP server's
`create_draft` tool. This does **not** send anything -- that server has no
send tool by design, so a human opens each draft in Gmail and sends it
manually. `sent` records the created draft's `gmail_thread_id` so a future
"Run B" node can look up replies via `search_threads`/`get_thread`.

Requires the connected account's token to include `gmail.compose` (see
`backend/gmail_oauth.py`) -- reconnect via the frontend if you connected
before that scope was added.

In [ ]:
async def sender_agent(state: RunAState) -> dict:
    """Sender node: creates a Gmail draft (via the `gmail` MCP server's
    create_draft tool) per approved draft. Does not send -- a human sends
    each draft manually from Gmail.
    """
    gmail_tools = {t.name: t for t in await client.get_tools(server_name="gmail")}
    create_draft_tool = gmail_tools["create_draft"]

    sent: list[dict] = []
    errors: list[str] = []

    drafts_to_send = state.get("approved_drafts") or state.get("drafts", [])

    for draft in drafts_to_send:
        try:
            raw = await create_draft_tool.ainvoke({
                "to": [draft.to_email],
                "subject": draft.subject,
                "body": draft.body,
            })
            payload = json.loads(raw[0]["text"])
            sent.append({
                "customer_id": draft.customer_id,
                "gmail_thread_id": payload.get("threadId"),
                "gmail_draft_id": payload.get("id"),
                "status": "drafted",
            })
        except Exception as exc:
            errors.append(f"{draft.customer_id}: draft creation failed - {exc}")

    return {
        "sent": sent,
        "errors": errors,
    }

## Build Graph

In [ ]:
graph_builder = StateGraph(RunAState)
graph_builder.add_node("Check_Weather",weather_monitor_agent)
graph_builder.add_node("Get_Customer",targeting_agent)
graph_builder.add_node("Get_Email_Draft",personalizer_agent)
graph_builder.add_node("Send_Draft",sender_agent)

graph_builder.add_edge(START,"Check_Weather")
graph_builder.add_edge("Check_Weather","Get_Customer")
graph_builder.add_edge("Get_Customer","Get_Email_Draft")
graph_builder.add_edge("Get_Email_Draft","Send_Draft")
graph_builder.add_edge("Send_Draft",END)

graph=graph_builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [13]:
# Live OpenWeather data has no qualifying storm right now in any DB metro
# (Dallas, Austin, San Antonio, Denver, Colorado Springs, Tampa, Orlando,
# Miami, Jacksonville, Houston all checked clear/calm). To exercise the real
# DB lookup + real GPT draft generation end-to-end, simulate a qualifying
# StormEvent for one DB city and run it through the same node functions the
# graph uses (mirrors the manual test pattern used above for each node).

STORM_CITY = "Dallas"  # any DB metro works: Dallas, Austin, San Antonio, Denver,
                        # Colorado Springs, Tampa, Orlando, Miami, Jacksonville, Houston

simulated_event = StormEvent(
    city=STORM_CITY,
    condition_code=200,  # thunderstorm
    condition_text="thunderstorm with heavy rain",
    wind_speed=45,
    wind_units="mph",
    date=datetime.now(timezone.utc).date().isoformat(),
)
assert simulated_event.qualifies

monitor_result = {
    "storm_events": [simulated_event],
    "qualifying_cities": [STORM_CITY],
    "errors": [],
}

targeting_result = await targeting_agent({"qualifying_cities": monitor_result["qualifying_cities"]})
personalizer_result = await personalizer_agent({
    "leads": targeting_result["leads"],
    "storm_events": monitor_result["storm_events"],
})

response = {**monitor_result, **targeting_result, **personalizer_result}

print("qualifying_cities:", response["qualifying_cities"])
print("leads:", len(response["leads"]))
print("drafts:", len(response["drafts"]))
if response["drafts"]:
    print(response["drafts"][0])


qualifying_cities: ['Dallas']
leads: 0
drafts: 0


# try it: draft + read, end to end

Runs `sender_agent` on the simulated drafts from above (creates real Gmail
drafts -- check the connected account's Drafts folder), then demonstrates
reading mail back via the `gmail` MCP server's `search_threads` tool (e.g.
what a future "Run B" reply-checker would use).

`backend.gmail_client.send_email()` still exists if you ever want the agent
to send directly via the REST API instead of drafting -- just not used here.

In [ ]:
sender_result = await sender_agent({"drafts": response["drafts"]})
print("drafted:", sender_result["sent"])
if sender_result["errors"]:
    print("errors:", sender_result["errors"])

In [ ]:
# example: read mail back via MCP (what a "Run B" reply-checker would use)
gmail_tools = {t.name: t for t in await client.get_tools(server_name="gmail")}
search_threads_tool = gmail_tools["search_threads"]

if response["drafts"]:
    lead_email = response["drafts"][0].to_email
    raw = await search_threads_tool.ainvoke({"query": f"to:{lead_email}", "pageSize": 5})
    print(json.loads(raw[0]["text"]))
else:
    print("no drafts to look up")